# Februar-Stundendaten der letzten 20 Jahre mit Plotly vergleichen

Dieses Notebook baut auf dem einfachen Wetterdaten-Beispiel auf, verwendet aber bewusst den Geosphere-Stundendatensatz.

Ziel ist ein interaktiver Vergleichsplot, in dem die Februare der letzten Jahre uebereinandergelegt werden.

Wichtige Unterschiede zum einfacheren Notebook:

1. Wir verwenden `klima-v2-1h` statt `klima-v2-10min`
2. Vor dem Laden pruefen wir, welche Lienz-Stationen im Stundendatensatz verfuegbar sind
3. Der Untersuchungszeitraum wird automatisch auf die real verfuegbaren Stationsjahre begrenzt
4. Die Daten werden kompakt als `csv.gz` und interaktive HTML-Datei gespeichert

## Datenquelle

Wir verwenden den offiziellen Stundendatensatz von Geosphere Austria:

- Datensatzseite: `https://data.hub.geosphere.at/dataset/klima-v2-1h`
- Metadaten: `https://dataset.api.hub.geosphere.at/v1/station/historical/klima-v2-1h/metadata`
- Messdaten: `https://dataset.api.hub.geosphere.at/v1/station/historical/klima-v2-1h`

Die API liefert bereits Stundenwerte. Im Notebook werden sie trotzdem noch auf ein sauberes Stundenraster normalisiert, damit auch bei unvollstaendigen oder leicht unregelmaessigen Reihen ein konsistenter Vergleich moeglich bleibt.

In [9]:
from datetime import datetime
from pathlib import Path
from urllib.parse import urlencode
from urllib.request import urlopen
import json

import pandas as pd
import plotly.graph_objects as go

pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 160)

print("Bibliotheken geladen.")
print("Dieses Notebook verwendet pandas fuer die Aufbereitung und Plotly fuer den interaktiven Vergleichsplot.")

Bibliotheken geladen.
Dieses Notebook verwendet pandas fuer die Aufbereitung und Plotly fuer den interaktiven Vergleichsplot.


## 1. Konfiguration

Standardmaessig wird die Station `Lienz` und der Parameter `TL` verwendet. `TL` steht fuer Lufttemperatur. Im Notebook werden Parameter-Codes intern case-insensitive behandelt.

In [10]:
STATION_NAME = "Lienz"
PARAMETER_CODE = "TL"
REQUESTED_YEARS = 20

# Falls das Notebook vor Ende Februar ausgefuehrt wird, verwenden wir nur vollstaendige Februare bis inklusive des Vorjahres.
reference_now = datetime.now()
last_complete_february_year = reference_now.year if reference_now.month > 2 else reference_now.year - 1

NOTEBOOK_DIR = Path.cwd()
DATA_DIR = NOTEBOOK_DIR / "data"
PLOTS_DIR = NOTEBOOK_DIR / "plots"

DATA_DIR.mkdir(parents=True, exist_ok=True)
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Station: {STATION_NAME}")
print(f"Parameter-Code: {PARAMETER_CODE}")
print(f"Gewuenschte Anzahl Februare: {REQUESTED_YEARS}")
print(f"Letzter vollstaendiger Februar: {last_complete_february_year}")

Station: Lienz
Parameter-Code: TL
Gewuenschte Anzahl Februare: 20
Letzter vollstaendiger Februar: 2026


## 2. Hilfsfunktionen

Die Hilfsfunktionen kuemmern sich um:

- API-Aufrufe
- Auswahl der passenden Lienz-Stationen im Stundendatensatz
- automatische Begrenzung des Untersuchungszeitraums
- Laden eines Februars fuer genau die Station, die in diesem Jahr gueltig ist

In [11]:
DATASET_SLUG = "klima-v2-1h"
META_URL = f"https://dataset.api.hub.geosphere.at/v1/station/historical/{DATASET_SLUG}/metadata"
DATA_URL = f"https://dataset.api.hub.geosphere.at/v1/station/historical/{DATASET_SLUG}"


def load_json(url: str, params: dict | None = None) -> dict:
    """Lade JSON von einer URL und gib es als Python-Dictionary zurueck."""
    full_url = url if not params else f"{url}?{urlencode(params)}"
    with urlopen(full_url, timeout=180) as response:
        return json.load(response)


def load_station_metadata() -> pd.DataFrame:
    meta = load_json(META_URL)
    stations_df = pd.DataFrame(meta["stations"]).copy()
    stations_df["valid_from"] = pd.to_datetime(stations_df["valid_from"], utc=True)
    stations_df["valid_to"] = pd.to_datetime(stations_df["valid_to"], utc=True)
    return stations_df


def find_station_records(stations_df: pd.DataFrame, station_name: str) -> pd.DataFrame:
    matches = stations_df[stations_df["name"].str.contains(station_name, case=False, na=False)].copy()
    if matches.empty:
        raise ValueError(f"Keine Station fuer '{station_name}' gefunden.")
    return matches.sort_values(["type", "valid_from", "id"]).reset_index(drop=True)


def select_station_chain(matches: pd.DataFrame) -> pd.DataFrame:
    """Bevorzuge Individualstationen; falls es keine gibt, verwende die Combined-Station."""
    individual = matches[matches["type"] == "INDIVIDUAL"].copy()
    if not individual.empty:
        return individual.sort_values("valid_from").reset_index(drop=True)
    return matches[matches["type"] == "COMBINED"].copy().sort_values("valid_from").reset_index(drop=True)


def station_for_february(station_chain: pd.DataFrame, year: int) -> pd.Series | None:
    feb_start = pd.Timestamp(f"{year}-02-01T00:00:00", tz="UTC")
    feb_end = pd.Timestamp(f"{year}-03-01T00:00:00", tz="UTC")
    matches = station_chain[(station_chain["valid_from"] < feb_end) & (station_chain["valid_to"] >= feb_start)].copy()
    if matches.empty:
        return None
    return matches.sort_values("valid_from").iloc[-1]


def available_february_years(station_chain: pd.DataFrame, latest_year: int, requested_years: int) -> list[int]:
    available = []
    year = latest_year
    while year >= 1900 and len(available) < requested_years:
        if station_for_february(station_chain, year) is not None:
            available.append(year)
        year -= 1
    return sorted(available)


def normalize_parameter_code(parameter_code: str) -> str:
    """Geosphere liefert Parameter-Codes haeufig klein geschrieben zurueck."""
    return parameter_code.strip().lower()


def aggregation_for_parameter(parameter_code: str) -> str:
    """Waehle eine sinnvolle Stundenaggregation fuer den Parameter."""
    return "sum" if normalize_parameter_code(parameter_code) == "rr" else "mean"


def load_february_hourly(station_id: int, parameter_code: str, year: int) -> pd.DataFrame | None:
    parameter_code_normalized = normalize_parameter_code(parameter_code)
    start = f"{year}-02-01T00:00"
    end = f"{year}-03-01T00:00"
    query = {
        "station_ids": station_id,
        "parameters": parameter_code_normalized,
        "start": start,
        "end": end,
    }

    weather_json = load_json(DATA_URL, query)
    if not weather_json.get("features"):
        return None

    feature = weather_json["features"][0]
    parameter_dict = feature.get("properties", {}).get("parameters", {})
    if parameter_code_normalized not in parameter_dict:
        return None

    parameter_info = parameter_dict[parameter_code_normalized]
    parameter_label = f"{parameter_info['name']} ({parameter_info['unit']})"

    time_index = pd.to_datetime(weather_json["timestamps"], utc=True).tz_convert("Europe/Vienna")
    df_year = pd.DataFrame({"value": parameter_info["data"]}, index=time_index)
    df_year.index.name = "timestamp"
    df_year = df_year[df_year.index.month == 2].sort_index()
    if df_year.empty:
        return None

    rule = aggregation_for_parameter(parameter_code)
    if rule == "sum":
        hourly = df_year.resample("1h").sum(min_count=1)
    else:
        hourly = df_year.resample("1h").mean()

    hourly = hourly[hourly.index.month == 2].copy()
    hourly["year"] = year
    hourly["parameter"] = parameter_label
    hourly["hour_of_february"] = ((hourly.index.day - 1) * 24 + hourly.index.hour).astype(int)
    hourly["day_label"] = hourly.index.strftime("%d.%m %H:%M")
    hourly["station_id"] = station_id
    return hourly.reset_index()

## 3. Lienz-Stationen pruefen und Untersuchungszeitraum festlegen

Im Stundendatensatz existieren fuer `Lienz` mehrere Eintraege. Fuer die eigentliche Studie verwenden wir die Kette der `INDIVIDUAL`-Stationen und schauen zuerst, welche Februare damit tatsaechlich abgedeckt werden.

In [12]:
stations_df = load_station_metadata()
station_matches = find_station_records(stations_df, STATION_NAME)
station_chain = select_station_chain(station_matches)

display_columns = ["id", "type", "group_id", "name", "state", "altitude", "valid_from", "valid_to", "is_active"]
display(station_matches[display_columns])

YEARS = available_february_years(station_chain, latest_year=last_complete_february_year, requested_years=REQUESTED_YEARS)
if not YEARS:
    raise ValueError("Es wurden keine verfuegbaren Februare fuer diese Station gefunden.")

print(f"Verwendete Stationskette: {', '.join(str(x) for x in station_chain['id'])}")
print(f"Tatsaechlicher Studienzeitraum: {YEARS[0]} bis {YEARS[-1]} ({len(YEARS)} Februare)")

,id,type,group_id,name,state,altitude,valid_from,valid_to,is_active
0,55,COMBINED,NaN,Lienz,Tirol,661.0,1880-01-01 00:00:00+00:00,2100-12-31 00:00:00+00:00,True
1,17900,INDIVIDUAL,55.0,Lienz,Tirol,663.0,1934-06-20 00:00:00+00:00,1985-11-30 23:00:00+00:00,False
2,17901,INDIVIDUAL,55.0,Lienz,Tirol,661.0,1985-12-01 00:00:00+00:00,2100-12-31 00:00:00+00:00,True


Verwendete Stationskette: 17900, 17901
Tatsaechlicher Studienzeitraum: 2007 bis 2026 (20 Februare)


In [13]:
slug = STATION_NAME.lower().replace(' ', '_')
DATA_PATH = DATA_DIR / f"weather_february_hourly_{slug}_{PARAMETER_CODE.lower()}_{YEARS[0]}_{YEARS[-1]}.csv.gz"
HTML_PATH = PLOTS_DIR / f"weather_february_overlay_{slug}_{PARAMETER_CODE.lower()}_{YEARS[0]}_{YEARS[-1]}.html"

print(f"Datendatei: {DATA_PATH.resolve()}")
print(f"HTML-Plot: {HTML_PATH.resolve()}")

Datendatei: C:\Users\UlrichLehre\Documents\260324_LV3\notebooks\data\weather_february_hourly_lienz_tl_2007_2026.csv.gz
HTML-Plot: C:\Users\UlrichLehre\Documents\260324_LV3\notebooks\plots\weather_february_overlay_lienz_tl_2007_2026.html


## 4. Februare laden

Fuer jedes Jahr wird zuerst die passende Station aus der Stationskette bestimmt. Danach werden die Stundenwerte geladen.

Falls ein Parameter fuer ein Jahr doch nicht vorhanden ist, wird dieses Jahr sauber protokolliert und uebersprungen.

In [14]:
frames = []
skipped_years = []
station_usage_rows = []

for year in YEARS:
    station_row = station_for_february(station_chain, year)
    if station_row is None:
        skipped_years.append((year, "keine gueltige Station im Februar"))
        continue

    station_id = int(station_row["id"])
    print(f"Lade Februar {year} mit Station {station_id}...")
    df_year = load_february_hourly(station_id=station_id, parameter_code=PARAMETER_CODE, year=year)

    if df_year is None or df_year['value'].notna().sum() == 0:
        skipped_years.append((year, f"kein Parameter {normalize_parameter_code(PARAMETER_CODE)} verfuegbar"))
        continue

    station_usage_rows.append({
        "year": year,
        "station_id": station_id,
        "station_type": station_row["type"],
        "valid_from": station_row["valid_from"],
        "valid_to": station_row["valid_to"],
    })
    frames.append(df_year)

if not frames:
    raise ValueError(f"Fuer {PARAMETER_CODE} konnten keine Februar-Daten geladen werden.")

hourly_df = pd.concat(frames, ignore_index=True)
parameter_label = hourly_df["parameter"].iloc[0]
station_usage_df = pd.DataFrame(station_usage_rows)
loaded_years = sorted(hourly_df["year"].unique())

print("Daten wurden geladen.")
print(f"Tatsaechlich geladene Jahre: {loaded_years[0]} bis {loaded_years[-1]} ({len(loaded_years)} Februare)")
print(f"Zeilen gesamt: {len(hourly_df):,}")
print(f"Messgroesse: {parameter_label}")
if skipped_years:
    print("Uebersprungene Jahre:")
    for year, reason in skipped_years:
        print(f"- {year}: {reason}")

Lade Februar 2007 mit Station 17901...
Lade Februar 2008 mit Station 17901...
Lade Februar 2009 mit Station 17901...
Lade Februar 2010 mit Station 17901...
Lade Februar 2011 mit Station 17901...
Lade Februar 2012 mit Station 17901...
Lade Februar 2013 mit Station 17901...
Lade Februar 2014 mit Station 17901...
Lade Februar 2015 mit Station 17901...
Lade Februar 2016 mit Station 17901...
Lade Februar 2017 mit Station 17901...
Lade Februar 2018 mit Station 17901...
Lade Februar 2019 mit Station 17901...
Lade Februar 2020 mit Station 17901...
Lade Februar 2021 mit Station 17901...
Lade Februar 2022 mit Station 17901...
Lade Februar 2023 mit Station 17901...
Lade Februar 2024 mit Station 17901...
Lade Februar 2025 mit Station 17901...
Lade Februar 2026 mit Station 17901...
Daten wurden geladen.
Tatsaechlich geladene Jahre: 2007 bis 2026 (20 Februare)
Zeilen gesamt: 13,540
Messgroesse: Lufttemperatur 2m (°C)


## 5. Daten kurz ansehen

Hier sieht man sowohl die ersten Zeilen der eigentlichen Messwerte als auch die pro Jahr verwendete Station.

In [15]:
display(hourly_df.head())
display(station_usage_df)

summary_df = hourly_df.groupby("year", as_index=False).agg(
    rows=("value", "size"),
    missing_values=("value", lambda s: int(s.isna().sum())),
    minimum=("value", "min"),
    maximum=("value", "max"),
)

display(summary_df)

,timestamp,value,year,parameter,hour_of_february,day_label,station_id
0,2007-02-01 01:00:00+01:00,-4.2,2007,Lufttemperatur 2m (°C),1,01.02 01:00,17901
1,2007-02-01 02:00:00+01:00,-3.3,2007,Lufttemperatur 2m (°C),2,01.02 02:00,17901
2,2007-02-01 03:00:00+01:00,-4.0,2007,Lufttemperatur 2m (°C),3,01.02 03:00,17901
3,2007-02-01 04:00:00+01:00,-0.9,2007,Lufttemperatur 2m (°C),4,01.02 04:00,17901
4,2007-02-01 05:00:00+01:00,-1.9,2007,Lufttemperatur 2m (°C),5,01.02 05:00,17901


,year,station_id,station_type,valid_from,valid_to
0,2007,17901,INDIVIDUAL,1985-12-01 00:00:00+00:00,2100-12-31 00:00:00+00:00
1,2008,17901,INDIVIDUAL,1985-12-01 00:00:00+00:00,2100-12-31 00:00:00+00:00
2,2009,17901,INDIVIDUAL,1985-12-01 00:00:00+00:00,2100-12-31 00:00:00+00:00
3,2010,17901,INDIVIDUAL,1985-12-01 00:00:00+00:00,2100-12-31 00:00:00+00:00
4,2011,17901,INDIVIDUAL,1985-12-01 00:00:00+00:00,2100-12-31 00:00:00+00:00
5,2012,17901,INDIVIDUAL,1985-12-01 00:00:00+00:00,2100-12-31 00:00:00+00:00
6,2013,17901,INDIVIDUAL,1985-12-01 00:00:00+00:00,2100-12-31 00:00:00+00:00
7,2014,17901,INDIVIDUAL,1985-12-01 00:00:00+00:00,2100-12-31 00:00:00+00:00
8,2015,17901,INDIVIDUAL,1985-12-01 00:00:00+00:00,2100-12-31 00:00:00+00:00
9,2016,17901,INDIVIDUAL,1985-12-01 00:00:00+00:00,2100-12-31 00:00:00+00:00


,year,rows,missing_values,minimum,maximum
0,2007,671,0,-10.4,13.2
1,2008,695,0,-9.5,18.6
2,2009,671,0,-13.8,11.8
3,2010,671,0,-12.9,9.8
4,2011,671,0,-12.6,14.9
5,2012,695,0,-16.6,19.4
6,2013,671,0,-14.5,6.1
7,2014,671,0,-7.4,9.4
8,2015,671,0,-10.4,11.6
9,2016,695,0,-7.3,11.9


## 6. Grossen Plotly-Vergleichsplot erzeugen

Alle geladenen Februare werden uebereinandergelegt. Die x-Achse zeigt die Stunde innerhalb des Februars, nicht das absolute Datum.

Dadurch lassen sich Jahresverlaeufe direkt vergleichen. Das aktuellste Jahr wird etwas staerker hervorgehoben.

In [16]:
def build_overlay_figure(df: pd.DataFrame, station_label: str, parameter_label: str) -> go.Figure:
    fig = go.Figure()
    years_sorted = sorted(df["year"].unique())
    latest_year = max(years_sorted)

    for year in years_sorted:
        year_df = df[df["year"] == year].sort_values("hour_of_february")
        is_latest = year == latest_year
        fig.add_trace(
            go.Scattergl(
                x=year_df["hour_of_february"],
                y=year_df["value"],
                mode="lines",
                name=str(year),
                line={"width": 3 if is_latest else 1.25},
                opacity=1.0 if is_latest else 0.55,
                customdata=year_df[["day_label", "station_id"]],
                hovertemplate="Jahr %{fullData.name}<br>%{customdata[0]}<br>Wert: %{y:.2f}<br>Station-ID: %{customdata[1]}<extra></extra>",
            )
        )

    tick_values = [0, 7 * 24, 14 * 24, 21 * 24, 28 * 24]
    tick_labels = ["01.02", "08.02", "15.02", "22.02", "29.02"]

    fig.update_layout(
        title=f"{parameter_label} im Februar: {station_label}, verfuegbare letzte Jahre",
        template="plotly_white",
        height=850,
        hovermode="x unified",
        legend_title_text="Jahr",
        xaxis_title="Stunde innerhalb des Februars",
        yaxis_title=parameter_label,
    )
    fig.update_xaxes(tickmode="array", tickvals=tick_values, ticktext=tick_labels, showgrid=True)
    fig.update_yaxes(showgrid=True)
    return fig

In [17]:
fig = build_overlay_figure(hourly_df, station_label=STATION_NAME, parameter_label=parameter_label)
fig.show()

ValueError: Mime type rendering requires nbformat>=4.2.0 but it is not installed

## 7. Kompakt speichern

Gespeichert werden nur die aufbereiteten Stundenwerte sowie der interaktive HTML-Plot. Dadurch bleiben die Dateien deutlich kleiner als ein Export aller Rohdaten.

In [ ]:
hourly_export = hourly_df.copy()
if pd.api.types.is_datetime64tz_dtype(hourly_export["timestamp"]):
    hourly_export["timestamp"] = hourly_export["timestamp"].dt.tz_localize(None)

hourly_export.to_csv(DATA_PATH, index=False, compression="gzip")
fig.write_html(HTML_PATH, include_plotlyjs="cdn")

print("Export fertig.")
print(f"CSV.GZ: {DATA_PATH.resolve()}")
print(f"HTML:   {HTML_PATH.resolve()}")

## 8. Was leicht angepasst werden kann

- `STATION_NAME` aendern, um eine andere Station zu laden
- `PARAMETER_CODE` auf andere Groessen umstellen, zum Beispiel `RF`, `P`, `FFAM` oder `RR`
- `REQUESTED_YEARS` vergroessern oder verkleinern

Wenn fuer ein frueheres Jahr zwar eine gueltige Station existiert, der gewuenschte Parameter aber fehlt, wird das Jahr automatisch uebersprungen und im Notebook protokolliert.